# Weather Logger — A Mini Data Engineering Project

**Goal:** Build a small tool that calls a public weather API, models the data with a class, cleans/validates it, saves it to a file, handles errors gracefully, and is covered by basic tests.

**Dataset shape (6 fields, one row per API call):**
1. `city` (str)
2. `temperature` (float)
3. `humidity` (int)
4. `description` (str)
5. `wind_speed` (float)
6. `timestamp` (str, ISO format)

Every task below builds on the previous one — nothing here is a standalone exercise. By the end, running one function will call the API, validate the response, log it, save it to CSV, and be testable.

**Suggested API:** [Open-Meteo](https://open-meteo.com/) (free, no API key needed) or OpenWeatherMap (needs a free key). Open-Meteo is simpler to start with since there's no auth to deal with.


## Step 1 — Design the data model (OOP / Classes)

Create a `WeatherRecord` class that represents one row of your dataset.

- Attributes: `city`, `temperature`, `humidity`, `description`, `wind_speed`, `timestamp`.
- A `__init__` that takes these values.
- A `__repr__` or `__str__` so printing a record looks clean.
- A `to_dict()` method — you'll need this later for saving to CSV/JSON.
- (Optional, if you want to push OOP further) A `to_csv_row()` method, or a small `WeatherLog` class that holds a *list* of `WeatherRecord` objects and has methods like `add()`, `average_temperature()`, or `save_all()`.

This class is the object every later step will produce, clean, save, or test.


## Step 2 — Fetch data from the API (Calling API)

Write a function `fetch_weather(city_name)` that:

- Sends a request to your chosen weather API for a given city.
- Parses the JSON response.
- Extracts the 6 fields you need for `WeatherRecord`.
- Returns a `WeatherRecord` instance (using the class from Step 1) — not a raw dict.

This connects directly to Step 1: the API call's *only job* is to produce the object you already designed.


## Step 3 — Clean and validate the input (Regular Expressions)

Before calling the API, you'll ask the user for a city name. Real input is messy — extra spaces, numbers, symbols, mixed casing.

Write a function `clean_city_name(raw_input)` that:

- Uses regex to strip out anything that isn't a letter, space, or hyphen (rejects things like `"Paris123"` or `"New York!!"`).
- Trims extra whitespace.
- Capitalizes it properly (e.g., `"new   york"` → `"New York"`).
- Raises a `ValueError` (or returns `None`) if what's left is empty or invalid — this feeds directly into your error handling in Step 5.

This step sits *before* Step 2 in the pipeline: clean the city name first, then fetch its weather.


## Step 4 — Add logging and retries (Decorators)

Write a decorator `@retry_on_failure(times=3)` (or `@log_call`, or both) and apply it to your `fetch_weather` function from Step 2.

- `@log_call`: prints/logs the function name, arguments, and how long the call took.
- `@retry_on_failure`: if the API call fails (network error, timeout), automatically retries up to N times before giving up.

This is where decorators earn their place — you're not writing a toy example, you're wrapping the *real* API function from Step 2 to make it more reliable.


## Step 5 — Handle errors gracefully (Error Handling)

Now formalize the failure paths that Steps 2–4 created:

- Wrap the API call in `try/except` for connection errors, timeouts, and bad JSON.
- Catch the `ValueError` from Step 3 when a city name is invalid.
- Consider a custom exception class, e.g. `WeatherFetchError(Exception)`, raised when the retries in Step 4 are exhausted.
- Decide what the program does on failure: skip the city and continue, or log the error and stop.

Nothing new is introduced here — you're making the pipeline from Steps 1–4 resilient instead of letting it crash.


## Step 6 — Save and load the data (File Handling + Data Handling)

Write two functions:

- `save_records(records, filepath)` — takes a list of `WeatherRecord` objects, converts each with `to_dict()` (Step 1), and writes them to a CSV (or JSON) file.
- `load_records(filepath)` — reads the file back and reconstructs a list of `WeatherRecord` objects.

Then do something with the loaded data — e.g., compute the average temperature across all saved cities, or find the city with the highest humidity. This is your "data handling" piece: simple aggregation over the dataset you built.


## Step 7 — Put it all together (Modules & Packages)

Split your code into a small package structure instead of one giant notebook cell block:

```
weather_logger/
├── __init__.py
├── models.py       # WeatherRecord (Step 1)
├── api.py          # fetch_weather + decorators (Steps 2 & 4)
├── validation.py   # clean_city_name (Step 3)
├── storage.py       # save_records / load_records (Step 6)
└── errors.py        # WeatherFetchError (Step 5)
```

Then import from these modules in your notebook and run the full pipeline: clean input → fetch → save → load → summarize.


## Step 8 — Test it (Basic Testing)

Write a handful of `pytest` tests — these should target the *pure logic*, not the live API (don't hit the real network in tests):

- `test_clean_city_name`: valid input, messy input, invalid input (raises error).
- `test_weather_record_to_dict`: confirms `to_dict()` returns the right keys/values.
- `test_save_and_load_records`: write a couple of fake records to a temp file, read them back, check they match.
- `test_retry_decorator`: use a fake function that fails twice then succeeds, confirm the decorator retries correctly.

If you want to go one step further: use `unittest.mock` to fake the API call so `fetch_weather` can be tested without a real network request.


## Why this order matters

Each step depends on the one before it:

`clean_city_name` (3) → `fetch_weather` (2) → wrapped by decorators (4) → wrapped in try/except (5) → result saved with `save_records` (6) → everything organized into modules (7) → verified with tests (8) — all operating on the single `WeatherRecord` class from Step 1.

By the end you'll have one small, working, testable pipeline — not eight disconnected exercises.
